# Grouping and onset asynchrony

Two tasks. Headphones, low comfortable volume. Runtime → Run all.

In [ ]:
#@title Load
import importlib.util, subprocess, sys, math
from pathlib import Path

REF = "210f768eb1ac31a6df65099261c7e9fda6287a7b"
URL = "https://github.com/MeysamAmirsardari/SeqSFG_task.git"
IN_COLAB = importlib.util.find_spec("google.colab") is not None
if IN_COLAB:
    REPO = Path("/content/seqsfg_" + REF[:7])
    if not REPO.exists():
        subprocess.run(["git", "clone", "--quiet", URL, str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "--quiet", REF], check=True)
else:
    REPO = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "tcmr" / "stimulus.py").exists()), None)
    if REPO is None:
        raise RuntimeError("Open in Colab, or run from the SeqSFG repository.")
sys.path.insert(0, str(REPO))

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.colors import Normalize, LinearSegmentedColormap
import matplotlib.patheffects as pe
from scipy import stats
from IPython.display import Audio, HTML, display
from tcmr.config import Config as TcmrConfig, Condition, validate
from tcmr.stimulus import build as tcmr_build

import logging, io, wave, base64
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)
from matplotlib import font_manager
from scipy.signal import resample_poly
_have = {f.name for f in font_manager.fontManager.ttflist}
FONT = next(f for f in ["Helvetica Neue", "Arial", "Liberation Sans", "DejaVu Sans"] if f in _have)
plt.rcParams.update({"font.family": FONT, "font.size": 10,
                     "axes.linewidth": 0.8, "axes.spines.top": False, "axes.spines.right": False})
INK, ACC, SOFT, GREY, MARK = "#1f2933", "#2f5d8a", "#c9d6e3", "#9aa5b1", "#12c25a"
CMAP = LinearSegmentedColormap.from_list("level", ["#e6eef6", "#a9c8e2", "#5f9acb", "#2f5d8a", "#132a45"])


In [ ]:
#@title Stimuli
TCMR = validate(TcmrConfig.from_dict({'name': 'tcmr_exp1', 'sample_rate': 48000, 'channels': 2, 'device_match': 'Headphones', 'blocksize': 256, 'latency': 'low', 'bands_hz': [250.0, 500.0, 1000.0, 2000.0, 4000.0], 'target_band': 2, 'bandwidth_hz': 20.0, 'band_db': 60.0, 'burst_ms': 50.0, 'ramp_ms': 10.0, 'noi_ms': 200.0, 'n_precursors': 5, 'tone_hz': 1000.0, 'isi_ms': 400.0, 'min_gap_ms': 25.0, 'lead_ms': 80.0, 'conditions': [['co', 'sync', 0.0, 1], ['rand', 'sync', 0.0, 1], ['co', 'frozen', 5.0, 1], ['rand', 'frozen', 5.0, 1], ['co', 'frozen', 10.0, 1], ['rand', 'frozen', 10.0, 1], ['co', 'frozen', 20.0, 1], ['rand', 'frozen', 20.0, 1], ['co', 'frozen', 40.0, 1], ['rand', 'frozen', 40.0, 1], ['ref', 'sync', 0.0, 1]], 'start_db': 75.0, 'min_db': 20.0, 'max_db': 85.0, 'steps_db': [8.0, 4.0, 2.0], 'step_change_after': [2, 4], 'n_down': 2, 'trials_per_track': 32, 'lapse': 0.02, 'practice_levels_db': [80.0, 80.0, 78.0, 78.0, 75.0, 75.0, 72.0, 72.0], 'ratings': [], 'rating_reps': 0, 'ready_ms': 400, 'feedback_ms': 500, 'blank_ms': 600, 'break_every_rounds': 9, 'uncal_peak_60db': 0.02, 'burst_level': 'natural'}))
TINC = {'name': 'tinc_8db', 'sample_rate': 48000, 'channels': 2, 'device_match': 'Headphones', 'blocksize': 256, 'latency': 'low', 'f_b_hz': 1000.0, 'df_semitones': 15.0, 'tone_ms': 100.0, 'ramp_ms': 20.0, 'soa_ms': 200.0, 'n_tones': 12, 'delays_ms': [0.0, 35.0, 50.0, 71.0, 100.0], 'levels': 'common', 'level_db': 60.0, 'rove_db': 8.0, 'decoy_db': 8.0, 'decoy_counts': [1, 2], 'target_db': 8.0, 'target_from': 4, 'blocks_per_delay': 2, 'trials_per_block': 52, 'examples': [[0.0, 1], [0.0, 0], [100.0, 1], [100.0, 0]], 'practice': [[0.0, 12, 12.0], [0.0, 10], [100.0, 10]], 'feedback': True, 'pre_ms': 40.0, 'post_ms': 80.0, 'ready_ms': 400, 'feedback_ms': 500, 'blank_ms': 600, 'max_peak': 0.95, 'uncal_peak_60db': 0.02}
SR = TCMR.sample_rate

def tinc_build(delay_ms, change, seed, target_db=None):
    """tinc.stimulus.build for levels = "common" (the 8-dB task), sample for sample."""
    c = TINC
    rng = np.random.default_rng(list(seed))
    n = c["n_tones"]
    k = int(rng.choice(c["decoy_counts"]))
    rove = rng.uniform(-c["rove_db"], c["rove_db"], size=n)
    decoys = sorted(int(x) for x in rng.choice(n, size=k, replace=False))
    target = int(rng.integers(c["target_from"], n))
    ph_b = rng.uniform(0.0, 2.0 * math.pi, size=n)
    ph_a = rng.uniform(0.0, 2.0 * math.pi, size=n)
    t_idx = target if change else -1
    f_b = c["f_b_hz"]; f_a = f_b * 2.0 ** (-c["df_semitones"] / 12.0)
    pre, dur, ramp = c["pre_ms"] / 1000, c["tone_ms"] / 1000, c["ramp_ms"] / 1000
    longest = (n - 1) * c["soa_ms"] + max(c["delays_ms"]) + c["tone_ms"]
    N = int(round((c["pre_ms"] + longest + c["post_ms"]) / 1000.0 * c["sample_rate"]))
    m = int(round(dur * SR)); t = np.arange(m) / SR
    env = np.ones(m); on, off = t < ramp, t > dur - ramp
    env[on] = np.sin(0.5 * math.pi * t[on] / ramp) ** 2
    env[off] = np.sin(0.5 * math.pi * (dur - t[off]) / ramp) ** 2
    amp = lambda db: c["uncal_peak_60db"] * 10 ** ((db - 60.0) / 20)
    stem, tones = np.zeros((2, N)), []
    for i in range(n):
        s = c["decoy_db"] if i in decoys else 0.0
        la = c["level_db"] + rove[i] + s
        lb = la + ((c["target_db"] if target_db is None else target_db) if i == t_idx else 0.0)
        on_b = pre + i * c["soa_ms"] / 1000.0
        for row, f, onset, lv, p in ((0, f_b, on_b, lb, ph_b[i]), (1, f_a, on_b + delay_ms / 1000, la, ph_a[i])):
            i0 = int(round(onset * SR))
            stem[row, i0:i0 + m] += amp(lv) * env * np.sin(2 * math.pi * f * t + p)
            tones.append(("B" if row == 0 else "A", onset - pre, lv, row == 0 and i == t_idx))
    return stem.sum(0), tones, f_a, f_b

def runs(x):
    """(onset, offset) sample pairs of the non-silent runs in one band."""
    nz = np.r_[0, (np.abs(x) > 0).astype(int), 0]
    d = np.diff(nz)
    return list(zip(np.flatnonzero(d == 1), np.flatnonzero(d == -1)))


In [ ]:
#@title Plot and play
PLAY_SR = 16000                                    # every component is below 4.1 kHz

def wav_tag(x, label):
    y = resample_poly(np.asarray(x, float), 1, SR // PLAY_SR)
    pcm = (np.clip(y, -1, 1) * 32767).astype("<i2")   # no normalisation: levels stay as built
    b = io.BytesIO()
    with wave.open(b, "wb") as w:
        w.setnchannels(1); w.setsampwidth(2); w.setframerate(PLAY_SR); w.writeframes(pcm.tobytes())
    src = base64.b64encode(b.getvalue()).decode()
    return (f"<div style='font-size:12px;color:#1f2933;margin-bottom:2px'>{label}</div>"
            f"<audio controls preload='none' style='width:190px;height:30px' src='data:audio/wav;base64,{src}'></audio>")

def table(rows):
    tr = "".join("<tr>" + "".join(f"<td style='padding:4px 10px;vertical-align:bottom'>{c}</td>" for c in r) + "</tr>" for r in rows)
    display(HTML(f"<table style='border-collapse:collapse;margin:4px 0 18px 0'>{tr}</table>"))

def style(ax, labels, ticks):
    ax.set_yticks(ticks, labels); ax.tick_params(axis="y", length=0, labelsize=8)
    ax.tick_params(axis="x", labelsize=8); ax.spines["left"].set_visible(False)

HALO = [pe.Stroke(linewidth=7.5, foreground="white"), pe.Normal()]
NAMES = {"co": "Comodulated", "rand": "Independent", "ref": "1-kHz band alone"}

# ---- tcmr: sound 1 has the tone, sound 2 does not
def tcmr_pair(env, step, level_db, seed):
    cond = Condition(env, "sync" if step == 0 else "frozen", float(step))
    tr, buf, stem = tcmr_build(TCMR, cond, level_db, seed, n_intervals=2, signal_interval=1, stems=True)
    sounds = [buf[int(on * SR):int((off + 0.03) * SR), 0] for on, off in zip(tr.interval_on_s, tr.interval_off_s)]
    bursts = []
    for b in range(5):
        for i0, i1 in runs(stem[b]):
            t = i0 / SR
            iv = max(k for k, o in enumerate(tr.interval_on_s) if o <= t + 1e-9)
            bursts.append((iv, b, t - tr.interval_on_s[iv], 20 * np.log10(np.sqrt(np.mean(stem[b, i0:i1] ** 2)))))
    return dict(sounds=sounds, bursts=bursts, tone=tr.target_on_s[0] - tr.interval_on_s[0])

def tcmr_panel(ax, pair, iv, norm):
    for b in range(5):
        ax.axhline(b, color="0.92", lw=0.6, zorder=0)
    for k, b, t, db in pair["bursts"]:
        if k == iv:
            ax.add_patch(Rectangle((t, b - 0.2), TCMR.burst_ms / 1000, 0.4, fc=CMAP(norm(db)), ec="none"))
    if iv == 0:
        ax.plot([pair["tone"] - 0.006, pair["tone"] + TCMR.burst_ms / 1000 + 0.006], [2, 2], color=MARK,
                lw=4.5, solid_capstyle="butt", zorder=5, path_effects=HALO)
    ax.set_xlim(0, 1.16); ax.set_ylim(-0.6, 4.6)
    style(ax, ["0.25", "0.5", "1", "2", "4"], range(5))

def show_tcmr(step, envs=("co", "rand")):
    rows = [[tcmr_pair(e, step, L, [20261116, int(step), int(L), j]) for j, e in enumerate(envs)] for L in LEVELS]
    lv = np.array([x[3] for r in rows for t in r for x in t["bursts"]])
    norm = Normalize(lv.max() - 16, lv.max())
    ncol = 3 * len(envs) - 1
    fig = plt.figure(figsize=(6.4 * len(envs) + 0.6, 1.6 * len(LEVELS) + 0.9))
    gs = fig.add_gridspec(len(LEVELS), ncol, width_ratios=([1, 1, 0.22] * len(envs))[:ncol], wspace=0.12, hspace=0.45)
    for r, L in enumerate(LEVELS):
        for j, e in enumerate(envs):
            for iv in range(2):
                ax = fig.add_subplot(gs[r, 3 * j + iv])
                tcmr_panel(ax, rows[r][j], iv, norm)
                if iv:
                    ax.set_yticklabels([])
                if r == 0:
                    ax.set_title(NAMES[e] + ("\nwith tone" if iv == 0 else "\nno tone"), loc="left", fontsize=9.5,
                                 color=MARK if iv == 0 else INK)
                if iv == 0 and j == 0:
                    ax.set_ylabel(f"tone {L:g} dB", fontsize=9, color=INK)
                if r < len(LEVELS) - 1:
                    ax.set_xticklabels([])
                else:
                    ax.set_xlabel("Time (s)", fontsize=8)
    plt.show()
    body = []
    for r, L in enumerate(LEVELS):
        cells = []
        for e, t in zip(envs, rows[r]):
            cells += [wav_tag(t["sounds"][0], f"<b>{NAMES[e]}</b>, tone {L:g} dB"),
                      wav_tag(t["sounds"][1], f"<b>{NAMES[e]}</b>, no tone")]
        body.append(cells)
    table(body)

# ---- tinc: sound 1 has the change, sound 2 does not
def tinc_panel(ax, tones, f_a, f_b, norm):
    y = {"A": 0.0, "B": np.log2(f_b / f_a)}
    for v in y.values():
        ax.axhline(v, color="0.92", lw=0.6, zorder=0)
    for s, t, db, tgt in tones:
        ax.add_patch(Rectangle((t, y[s] - 0.2), TINC["tone_ms"] / 1000, 0.4, fc=CMAP(norm(db)), ec="none"))
        if tgt:
            ax.add_patch(Rectangle((t - 0.015, y[s] - 0.34), TINC["tone_ms"] / 1000 + 0.03, 0.68, fc="none", ec=MARK,
                                   lw=2.2, zorder=5, path_effects=[pe.Stroke(linewidth=4, foreground="white"), pe.Normal()]))
    ax.set_xlim(-0.03, 2.45); ax.set_ylim(-0.6, y["B"] + 0.6)
    style(ax, [f"A {f_a / 1000:.2f}", f"B {f_b / 1000:g}"], [0, y["B"]])

def show_tinc(delay):
    rows = [(size, [tinc_build(delay, k == 0, [20261116, int(delay), int(size), k], target_db=size) for k in range(2)])
            for size in CHANGES]
    lv = np.array([x[2] for _, p in rows for b in p for x in b[1]])
    norm = Normalize(lv.min(), lv.max())
    fig = plt.figure(figsize=(13, 1.0 * len(CHANGES) + 0.8))
    gs = fig.add_gridspec(len(CHANGES), 3, width_ratios=[1, 0.06, 1], wspace=0.05, hspace=0.55)
    for r, (size, pair) in enumerate(rows):
        for k in range(2):
            ax = fig.add_subplot(gs[r, 2 * k])
            tinc_panel(ax, pair[k][1], pair[k][2], pair[k][3], norm)
            if k:
                ax.set_yticklabels([])
            if r == 0:
                ax.set_title("with change" if k == 0 else "no change", loc="left", fontsize=9.5,
                             color=MARK if k == 0 else INK)
            if k == 0:
                ax.set_ylabel(f"+{size:g} dB", fontsize=9, color=INK)
            if r < len(CHANGES) - 1:
                ax.set_xticklabels([])
            else:
                ax.set_xlabel("Time (s)", fontsize=8)
    plt.show()
    table([[wav_tag(pair[0][0], f"<b>Change</b>, one B tone +{size:g} dB"), wav_tag(pair[1][0], "<b>No change</b>")]
           for size, pair in rows])


In [ ]:
#@title Settings
TONE_LEVELS = "78, 72, 66, 60, 54"  #@param {type:"string"}
CHANGE_SIZES = "16, 12, 8, 4"  #@param {type:"string"}
LEVELS = [float(v) for v in TONE_LEVELS.split(",")]
CHANGES = [float(v) for v in CHANGE_SIZES.split(",")]


## tcmr

Left: with the 1-kHz tone. Right: without. Bands at 60 dB; darker = louder.

In [ ]:
#@title ΔT = 0 ms
show_tcmr(0)


In [ ]:
#@title ΔT = 5 ms
show_tcmr(5)


In [ ]:
#@title ΔT = 10 ms
show_tcmr(10)


In [ ]:
#@title ΔT = 20 ms
show_tcmr(20)


In [ ]:
#@title ΔT = 40 ms
show_tcmr(40)


In [ ]:
#@title 1-kHz band alone
show_tcmr(0, envs=("ref",))


## tinc

Left: one B tone louder. Right: no change. A and B rise and fall together.

In [ ]:
#@title A delay 0 ms
show_tinc(0)


In [ ]:
#@title A delay 35 ms
show_tinc(35)


In [ ]:
#@title A delay 50 ms
show_tinc(50)


In [ ]:
#@title A delay 71 ms
show_tinc(71)


In [ ]:
#@title A delay 100 ms
show_tinc(100)


## P00

In [ ]:
#@title Results
STEPS = [0, 5, 10, 20, 40]
CMR = [5.085, 8.039, 3.837, 4.027, 0.966]
CMR_DRAWS = [np.array(v) for v in [[4.55, 3.61, 1.24, 4.64, 5.84, 5.59, 8.36, 2.32, 4.22, 6.14, 5.83, 5.37, 4.52, 5.0, 5.67, 2.67, 3.37, 1.87, 4.91, 3.58, 5.07, 4.85, 5.67, 6.61, -0.05, 3.94, 3.26, 8.52, 6.32, 5.24, 4.05, 5.69, 4.15, 4.81, 6.53, 4.79, 5.97, 6.12, 5.75, 5.43, 7.06, 3.79, 5.87, 7.72, 7.16, 4.66, 7.24, 5.26, 4.41, 4.03, 6.49, 3.04, 5.76, 5.02, 5.12, 5.46, 4.74, 5.52, 5.08, 6.82, 3.95, 4.22, 5.47, 3.53, 4.46, 5.39, 6.5, 4.86, 4.26, 4.4, 5.81, 6.91, 6.54, 6.41, 7.86, 2.98, 4.84, 5.63, 3.5, 3.36, 5.62, 4.88, 5.63, 4.58, 8.24, 5.07, 2.93, 4.91, 2.02, 5.6, 4.7, 3.04, 4.08, 5.29, 2.27, 5.68, 3.95, 5.94, 6.05, 3.36, 3.43, 7.52, 4.63, 4.97, 5.31, 5.33, 5.28, 2.88, 4.82, 3.32, 5.46, 4.7, 6.75, 5.53, 4.22, 2.61, 4.59, 1.48, 3.34, 7.39, 2.53, 3.29, 4.54, 1.45, 3.61, 7.73, 2.57, 3.76, 4.76, 5.35, 1.0, 3.21, 2.87, 6.01, 4.77, 3.02, 6.82, 5.1, 4.99, 4.99, 7.91, 6.62, 4.68, 3.21, 4.71, 7.53, 2.91, 2.46, 4.96, 1.59, 8.74, 5.42, 3.82, 3.94, 2.7, 8.16, 3.33, 5.41, 5.41, 5.7, 3.26, 2.48, 5.62, 3.79, 6.42, 5.7, 3.25, 7.96, 3.49, 8.66, 1.98, 6.98, 5.27, 7.08, 4.0, 3.18, 6.47, 6.51, 5.21, 3.96, 6.19, 5.89, 5.5, 5.51, 5.61, 3.05, 7.69, 3.5, 2.31, 3.95, 6.82, 3.65, 5.73, 4.41, 3.48, 2.71, 3.98, 8.37, 3.82, 4.58, 3.95, -0.45, 6.84, 6.86, 3.74, 8.28, 3.29, 5.04, 2.39, 4.04, 3.66, 2.99, 6.63, 3.52, 1.98, 3.36, 6.46, 6.35, 5.44, 5.21, 2.7, 3.62, 4.49, 4.73, 4.58, 3.85, 5.9, 5.61, 7.25, 5.42, 4.92, 2.39, 6.71, 4.84, 1.91, 8.15, 5.14, 4.56, 4.22, 3.9, 5.24, 4.65, 4.09, 7.48, 4.67, 4.58, 7.01, 6.83, 3.81, 6.89, 3.89, 6.41, 4.13, 5.19, 5.64, 3.57, 4.29, 5.0, 7.87, 4.84, 7.75, 6.45, 6.85, 5.7, 4.15, 5.92, 6.75, 5.89, 6.79, 3.55, 3.53, 5.01, 4.41, 9.34, 6.2, 4.6, 4.0, 7.14, 8.47, 4.31, 4.85, 7.17, 5.04, 5.39, 3.89, 3.16, 5.33, 3.39, 4.54, 8.29, 7.0, 4.6, 5.22, 1.68, 2.95, 5.33, 5.11, 4.15, 4.65, 7.81, 2.61, 5.53, 7.73, 4.17, 2.69, 8.27, 2.38, 8.91, 5.39, 3.62, 4.99, 2.7, 10.84, 3.5, 6.16, 5.43, 4.44, 2.34, 3.96, 2.22, 3.38, 5.32, 4.61, 4.25, 3.43, 1.11, 4.78, 6.39, 5.82, 3.05, 3.45, 1.67, 5.06, 7.31, 7.15, 3.95, 5.14, 3.01, 5.81, 3.02, 4.17, 4.02, 5.87, 6.43, 3.04, 5.71, 5.34, 2.72, 7.43, 6.91, 4.6, 5.2, 3.52, 6.17, 5.85, 9.59, 4.66, 5.22, 4.39, 4.57, 4.45, 4.54, 5.4, 6.12, 5.1, 5.85, 8.22, 5.88, 2.7, 2.7, 4.28, 3.73, 5.92, 4.56, 8.49, 4.09, 5.57, 6.38, 6.06, 3.65, 6.34, 4.45, 5.68, 5.51, 2.85, 6.76, 5.14, 4.42, 5.25, 3.76, 6.4, 5.81, 7.59, 3.92, 3.0, 5.66, 5.53, 6.88, 4.07, 7.41, 4.2, 6.07, 3.27, 4.11, 5.99, 5.21, 5.13, 3.16, 6.01, 5.83, 3.28, 4.09, 3.57, 6.56, 6.84, 4.89, 1.9, 6.4, 5.03, 6.83, 4.37, 4.9, 5.89, 5.96, 6.79, 6.49, 4.61, 5.69, 4.42, 5.83, 10.61, 3.73, 3.44, 3.52, 6.75, 4.7, 6.04, 5.89, 5.62, 2.49, 5.05, 5.81, 5.71, 3.06, 5.74, 6.18, 3.07, 4.92, 4.95, 4.29, 2.11, 4.44, 3.81, 5.48, 5.01, 3.71, 7.45, 4.13, 3.44, 6.02, 5.01, 4.88, 6.72, 5.49, 6.55, 3.43, 1.79, 5.27, 6.58, 5.5, 1.33, 6.87, 3.3, 8.13, 5.49, 5.11, 3.52, 4.38, 6.95, 8.08, 4.8, 7.38, 6.41, 3.33, 4.33, 2.41, 7.03, 7.58, 3.18, 1.14, 5.93, 4.66, 6.45, 5.81, 6.25, 6.31, 9.31, 5.62, 6.53, 2.07, 4.11, 5.95, 6.2, 6.21, 6.61, 6.91, 10.01, 3.84, 2.27, 3.53, 4.77, 6.68, 3.62, 3.68, 6.41, 4.11, 8.59, 8.02, 4.92, 5.6, 5.32, 5.54, 6.78, 2.22, 5.41, 8.68, 5.76, 2.94, 2.35, 4.92, 7.08, 7.06, 4.36, 5.05, 5.35, 4.52, 5.89, 4.7, 7.77, 5.46, 4.37, 4.8, 5.09, 5.65, 6.75, 3.75, 4.79, 4.66, 1.72, 3.56, 3.05, 4.79, 5.88, 7.56, 5.3, 4.28, 3.02, 3.41, 6.59, 4.91, 4.55, 5.29, 4.42, 4.09, 6.46, 2.6, 4.81, 4.23, 6.65, 6.22, 4.23, 7.53, 8.18, 6.5, 4.67, 3.92, 1.71, 3.51, 5.33, 4.62, 1.76, 6.09, 5.37, 4.93, 8.01, 4.52, 5.22, 5.84, 5.63, 5.46, 5.24, 4.67, 3.03, 4.72, 6.51, 4.16, 3.13, 5.42, 7.26, 4.51], [10.49, 3.18, 8.94, 9.3, 8.57, 5.66, 11.59, 6.73, 4.81, 10.7, 7.8, 10.41, 7.77, 7.78, 7.98, 3.16, 12.29, 10.57, 10.53, 8.64, 7.15, 7.91, 5.66, 8.38, 8.24, 5.17, 9.91, 7.15, 11.56, 8.87, 11.46, 10.24, 6.27, 5.67, 8.21, 6.19, 7.48, 10.63, 10.01, 6.25, 9.81, 11.83, 9.0, 5.78, 10.76, 6.95, 9.4, 10.41, 9.1, 9.11, 7.77, 5.91, 8.14, 12.51, 7.49, 9.54, 11.62, 8.97, 5.34, 8.5, 11.07, 11.27, 7.59, 9.98, 9.79, 8.61, 11.26, 8.79, 3.65, 7.16, 7.14, 9.21, 9.32, 7.57, 6.31, 5.82, 13.22, 9.54, 4.56, 4.08, 8.85, 14.46, 9.55, 9.76, 6.51, 6.59, 4.88, 8.6, 6.06, 11.7, 7.2, 7.24, 7.77, 6.42, 5.63, 3.3, 11.6, 3.82, 7.25, 7.6, 7.31, 8.84, 7.68, 7.81, 9.67, 9.05, 6.24, 6.18, 10.91, 7.74, 6.83, 11.51, 3.24, 11.03, 4.85, 6.42, 6.42, 5.3, 11.87, 13.09, 7.51, 11.0, 10.53, 7.13, 6.4, 6.32, 8.58, 8.37, 10.22, 6.3, 9.39, 8.02, 6.81, 9.9, 6.25, 9.99, 8.22, 6.39, 8.39, 7.68, 3.29, 7.84, 9.23, 8.78, 5.34, 8.2, 8.95, 9.84, 9.41, 6.81, 5.33, 7.21, 9.99, 12.38, 8.62, 13.09, 5.35, 8.41, 7.16, 5.6, 8.02, 6.92, 9.68, 4.37, 10.88, 11.22, 4.7, 8.54, 7.61, 6.6, 7.83, 8.63, 6.99, 7.73, 7.11, 8.43, 10.19, 11.8, 6.63, 8.82, 2.67, 9.16, 6.41, 1.91, 6.47, 7.21, 7.36, 7.07, 5.52, 6.72, 10.24, 7.72, 4.89, 6.85, 24.88, 3.35, 7.33, 9.28, 7.38, 9.11, 5.04, 8.67, 10.47, 10.41, 7.3, 10.94, 9.28, 7.79, 17.17, 5.86, 7.21, 3.71, 5.77, 4.43, 4.15, 13.31, 11.28, 5.12, 6.36, 5.83, 10.03, 9.93, 6.66, 12.46, 8.59, 9.19, 7.47, 6.66, 9.85, 7.48, 6.46, 3.98, 3.05, 6.55, 11.05, 8.76, 10.98, 11.51, 11.64, 6.22, 6.61, 7.78, 6.94, 11.23, 5.52, 8.06, 8.16, 9.37, 7.08, 6.6, 6.61, 9.47, 9.66, 6.14, 11.46, 7.51, 6.76, 9.56, 8.87, 9.07, 6.85, 12.0, 9.76, 9.81, 13.96, 6.36, 4.91, 4.08, 5.28, 9.62, 10.41, 5.46, 9.53, 10.65, 7.98, 9.23, 8.71, 3.84, 10.33, 13.05, 5.99, 9.68, 10.83, 9.4, 6.3, 8.1, 5.72, 10.26, 8.63, 6.53, 6.37, 8.03, 7.98, 7.48, 9.4, 13.46, 6.72, 9.79, 10.96, 9.97, 8.57, 6.41, 7.17, 8.19, 11.25, 6.74, 7.39, 6.38, 5.94, 5.38, 7.49, 10.93, 8.05, 6.97, 4.49, 9.32, 7.74, 5.12, 11.63, 8.85, 9.15, 8.09, 10.63, 6.96, 5.82, 10.09, 8.3, 8.92, 5.48, 7.02, 9.05, 7.31, 6.65, 12.52, 8.21, 3.2, 4.14, 11.0, 6.27, 9.68, 10.72, 6.65, 16.86, 7.01, 9.8, 10.31, 6.21, 9.21, 12.77, 6.14, 10.65, 8.17, 5.09, 8.76, 13.62, 9.72, 7.42, 9.27, 9.78, 8.48, 15.44, 9.35, 2.02, 11.55, 3.86, 7.99, 8.9, 4.36, 9.05, 8.77, 6.55, 7.13, 3.95, 6.78, 6.58, 5.74, 7.74, 8.92, 11.57, 6.86, 8.59, 8.54, 9.2, 8.17, 12.2, 10.86, 7.84, 8.62, 6.64, 5.33, 4.42, 4.62, 9.07, 7.85, 9.25, 12.37, 11.05, 4.08, 10.31, 6.89, 6.59, 8.1, 8.1, 4.01, 7.02, 8.53, 7.27, 6.75, 9.66, 7.43, 10.33, 9.29, 7.08, 5.09, 9.89, 2.89, 8.28, 4.1, 5.58, 7.22, 7.07, 4.53, 10.99, 11.21, 8.26, 6.8, 7.94, 7.93, 9.19, 5.43, 9.16, 8.19, 6.55, 7.8, 11.89, 9.73, 8.91, 6.09, 7.58, 7.7, 6.2, 7.46, 5.98, 7.39, 9.33, 12.06, 10.83, 9.7, 9.0, 4.49, 8.51, 9.46, 3.24, 11.38, 7.03, 9.63, 8.86, 6.15, -8.82, 9.36, 8.73, 6.53, 7.27, 5.38, 3.19, 7.61, 9.28, 5.67, 3.33, 12.25, 9.2, 6.24, 9.41, 11.67, 7.83, 5.22, 5.68, 5.54, 8.51, 7.56, 11.72, 3.83, 5.11, 11.58, 12.16, 7.92, 7.8, 5.27, 7.61, 8.76, 6.81, 7.85, 6.92, 6.86, 14.83, 6.24, 5.45, 9.38, 5.46, 10.41, 7.52, 12.16, 5.3, 4.63, 7.4, 9.11, 10.07, 6.85, 9.12, 3.4, 10.68, 5.03, 10.05, 8.19, 9.86, 2.78, 6.09, 5.45, 12.98, 6.03, 6.66, 10.94, 9.03, 8.75, 8.84, 5.31, 4.75, 9.28, 10.87, 7.13, 13.31, 12.26, 6.45, -1.01, 6.99, 7.97, 7.4, 9.49, 8.89, 8.28, 7.51, 8.93, 8.56, 6.28, 4.98, 8.21, 13.75, 8.04, 5.22, 10.75, 6.95, 4.8, 6.61, 8.91, 4.8, 10.2, 10.0, 9.8, 7.01, 2.89, 6.48, 9.02, 7.9, 5.96, 5.34, 7.29, 10.86, 7.43, 7.58, 9.34, 5.15, 16.97, 1.98, 10.92, 8.32, 14.09, 8.01, 4.67, 1.53, 6.94, 5.2, 5.14, 7.52, 4.61, 6.3, 8.21, 8.73, 7.78, 9.19, 7.26, 6.04, 7.05, 4.28, 11.87, 9.23, 8.47, 12.6, 9.57, 7.97, 7.98], [5.15, 2.01, 4.17, 1.43, 1.29, 4.11, 1.78, 3.72, -0.35, 1.49, 1.75, 5.29, 2.51, 5.12, 3.89, 6.4, 5.2, 5.38, 4.63, 4.81, 3.87, 5.56, 3.53, 5.83, 3.2, 2.33, 0.56, 4.56, 3.54, 7.48, 1.3, 3.86, 2.8, 4.69, 5.8, 5.61, 9.31, 5.33, 3.75, 4.46, 1.1, 2.7, 1.98, 2.98, 4.32, 4.89, 1.92, 2.09, 1.51, 2.09, 1.46, 3.37, 2.94, 3.13, 1.41, 3.42, 2.09, 3.56, 4.76, 2.09, 2.77, 5.58, 2.97, 1.86, 4.56, 4.93, 3.49, 5.42, 1.62, 2.78, 2.28, 2.74, 5.52, 6.31, 6.69, 3.51, 5.27, 3.83, 2.61, 5.6, 5.22, 6.07, 4.4, 1.16, 0.98, 3.4, 5.57, 3.12, 2.73, 6.55, 4.48, 4.35, 2.06, 4.67, 4.47, 4.93, 3.84, 3.66, 1.68, 1.27, 2.7, 3.13, 3.47, 0.47, 3.92, 2.75, 4.25, 6.27, 4.13, 3.62, 7.22, 4.71, 4.29, 5.88, 2.56, 2.22, 5.68, 4.5, -0.1, 5.03, 4.97, 2.88, 5.23, 3.42, 5.19, 2.64, 5.37, 5.94, 3.45, 6.69, 2.25, 5.82, 4.42, 3.96, 1.07, 3.06, 5.28, 7.36, 5.35, 1.78, 3.02, 3.55, 2.95, 2.21, 0.5, 6.03, 5.63, 3.68, 1.1, 2.14, 5.2, 4.01, 6.31, 1.9, 1.38, 2.85, 3.29, 1.9, 3.42, 0.31, 3.42, 3.33, 4.03, 5.95, 1.82, 2.68, 3.51, 2.85, 1.82, 7.38, 4.41, 3.69, 5.65, 2.24, 3.74, 0.66, 4.68, 7.33, 3.38, 1.68, 5.39, 4.08, 6.7, 2.25, 6.95, 5.87, 1.21, 5.23, 3.22, 3.73, 3.7, 3.72, 5.63, -0.86, 3.09, 5.55, 5.5, 3.36, 4.08, 4.5, 4.18, 6.83, 2.64, 3.7, 4.77, 4.93, 4.83, 4.16, 4.0, 1.65, 0.13, 5.4, 2.8, 6.13, 5.07, 4.92, 4.05, 4.44, 1.52, 2.73, 2.46, 5.51, 4.85, 7.06, 5.71, 1.61, 1.86, 4.14, 6.74, 5.82, 5.98, 2.06, 4.99, 4.27, 4.56, 5.36, 4.1, 3.93, 5.85, 6.39, 1.95, 0.9, 5.25, 4.91, 1.41, 4.3, 6.1, 0.54, 4.43, 6.5, 2.46, 4.49, 3.1, 3.03, 6.29, 6.02, 3.3, 6.37, 4.59, 6.81, 3.34, 4.86, 2.05, 2.87, 2.26, 4.76, 4.36, 4.51, 4.9, 3.36, 0.3, 4.46, 6.6, 6.71, 5.34, 1.89, 4.84, 4.74, 5.9, 3.89, 4.5, 4.8, 4.03, 8.18, 2.96, 5.76, 5.23, 0.33, 1.38, 5.67, 3.57, 2.54, 2.88, 2.26, 3.13, 3.31, 3.97, 5.95, 3.96, 6.23, 2.68, 5.92, 4.77, 3.44, 8.28, 5.82, 5.99, 4.54, 4.46, 3.46, 6.08, 3.71, 4.46, 5.51, 3.1, 4.07, 4.16, 4.47, 5.35, 3.61, 3.93, 6.15, 5.15, 2.9, 1.16, 3.35, 4.86, 4.99, 4.24, 4.54, 4.25, 4.52, 2.34, 6.2, 3.07, 3.47, 5.17, 3.08, 7.06, 1.02, 6.25, 2.41, 0.97, 4.37, 4.27, 3.88, 3.87, 6.42, 5.02, 6.38, 4.4, 5.38, 3.91, 4.21, 3.84, 3.03, 2.83, 4.31, 4.34, 2.69, 2.83, 1.26, 6.04, 1.57, 3.36, 1.74, 4.94, 3.02, 4.4, 1.65, 2.75, 6.74, 5.45, 0.12, 2.95, 8.36, 4.12, 1.49, 7.95, 2.24, 5.22, 5.95, 3.95, 2.57, 1.21, 4.16, 3.74, 1.89, 6.0, 4.69, 3.91, 2.98, 2.41, 5.52, 2.31, 1.97, 6.29, 4.28, 4.68, 0.4, -0.24, 4.29, 5.39, 3.43, 5.33, 1.82, 5.04, 3.43, 6.57, 4.78, 1.61, 4.85, 2.31, 3.04, 2.9, 3.33, 6.35, 2.13, 3.08, 2.64, 2.79, 2.14, 2.6, 2.41, 2.49, 4.0, 3.84, 3.86, 5.23, 1.76, 2.98, 4.77, 0.41, 5.01, 5.77, 5.86, 2.67, 2.32, 5.0, 6.24, 3.45, 3.61, 3.99, 4.26, 3.92, 4.18, 7.08, 3.82, 7.42, 5.11, 2.05, 3.77, 3.72, 3.99, 1.81, 5.76, 3.06, 5.31, 2.86, 5.94, 4.65, 5.56, 4.85, 0.53, 2.68, 4.39, 5.25, 5.33, 5.3, 5.26, 4.83, 6.63, 5.87, 4.65, 3.46, 3.44, 4.32, 4.78, 4.56, 4.32, 4.46, 2.92, 5.62, 2.64, 2.3, 6.91, 0.95, 2.96, 7.07, 5.74, 5.21, -0.92, 1.49, 5.82, 5.13, 1.18, 1.39, 4.46, 3.7, 3.79, 5.76, 1.97, 3.95, 3.61, 5.97, 4.81, 1.62, 4.69, 4.69, 3.4, 4.95, 2.94, 1.72, 1.44, 4.15, 1.71, 4.72, -0.57, 2.42, 4.69, 4.18, 3.29, 3.86, 1.74, 4.48, 1.35, 3.52, 2.48, 4.89, 4.4, 5.26, 5.15, 5.73, 4.79, 6.21, 5.47, 6.22, 3.02, 3.17, 2.06, 3.62, 4.19, 4.07, 3.25, 5.67, 1.84, 5.75, 3.19, 2.81, 4.14, 5.67, 1.86, 0.05, 1.61, 6.33, 3.53, 3.88, 6.44, 4.3, 1.85, 0.85, 4.66, 6.48, 4.02, 3.04, -0.1, 4.19, 5.26, 6.69, 1.96, 6.55, 4.59, 1.08, 5.06, 1.79, 5.17, 4.37, 7.73, 2.5, 4.93, 2.08, 2.51, 2.57, 3.82, 2.07, 7.41, 2.36, 5.23, 2.94, 5.36, 5.65, 4.94, 3.13, 3.98, 6.48, 3.4, 6.0, 1.76, 6.18, 7.71], [3.98, 4.63, 2.05, 8.99, 0.99, 6.05, 1.29, 6.33, 9.35, 5.95, 2.87, 4.37, 5.35, 4.9, 3.68, 1.18, 5.05, 6.43, 2.7, 3.79, 3.45, 3.24, 0.87, 6.66, 2.32, 5.59, 5.25, 4.69, 1.58, 2.72, 3.55, 3.29, 3.38, 3.85, 2.7, 2.44, 3.71, 3.63, 6.51, -0.06, 3.01, 5.67, 6.85, 4.2, 4.51, 2.76, 3.22, 4.73, 3.22, 2.32, 1.48, 3.65, 6.2, 3.21, 3.29, 7.36, 2.99, 3.23, 4.38, 5.68, 1.84, 5.83, 3.67, 3.9, 4.39, 3.5, 3.87, 4.53, 3.9, 5.09, 3.26, 3.83, 7.95, 4.83, 3.35, 4.53, 2.85, 5.42, 4.12, 3.72, -0.36, 4.61, 4.41, 3.87, 6.51, 4.5, -0.35, 4.6, 2.53, 2.37, 5.16, 4.37, 4.54, 4.48, 4.59, 5.11, 5.8, 5.58, 4.63, 2.45, 4.52, 6.59, 4.67, 5.53, 2.67, 3.0, 2.79, 5.53, 4.42, 3.75, 1.15, 2.82, 1.59, 3.0, 5.13, 3.26, 4.54, 4.1, 4.86, 7.25, 0.83, 1.99, 4.33, 3.86, 5.02, 2.88, 4.03, 6.64, 4.96, 5.12, 0.56, 2.15, 2.68, 1.78, 4.3, 1.0, 4.2, 4.01, 4.7, 4.23, 2.1, 2.72, 6.02, 4.77, 4.2, 4.65, 2.31, 2.14, 4.48, 5.44, 4.86, 5.23, 3.43, 7.23, 4.16, 1.74, 3.34, 4.43, 4.0, 2.24, 6.95, 3.36, 4.08, 0.41, 6.23, 5.78, 3.48, 0.49, 1.72, 3.79, 4.82, 4.28, 2.67, 3.22, 3.05, 6.46, 1.67, 3.9, 0.22, 6.92, 6.24, 2.73, 5.04, 3.94, 5.28, 3.88, 4.68, 5.21, 4.36, 4.94, 4.69, 3.02, 4.68, 4.6, 3.23, 4.58, 6.35, 5.98, 2.91, 2.83, 2.95, 4.75, 3.99, 4.79, 6.02, -0.29, 3.97, 5.01, 7.92, 4.44, 6.36, 4.92, 5.96, 4.63, 4.98, 2.41, 3.62, 0.39, 4.44, 5.11, 4.2, 4.77, 5.23, 1.06, 3.4, 5.6, 3.58, 3.7, 2.23, 5.5, 2.64, 5.3, 4.66, 5.02, 0.22, 6.19, 3.47, 4.35, 4.17, 4.01, 4.1, 2.95, 3.06, 3.52, 2.69, 3.29, 1.91, 4.25, 4.85, 2.75, 2.06, 2.65, 0.25, 4.33, 4.13, 3.32, 3.61, 2.64, 4.5, 4.78, 3.06, 5.66, 0.67, 6.78, 3.53, 0.82, 5.71, 4.22, 3.55, 4.37, 4.79, 3.86, 4.41, 3.38, 4.01, 2.98, 3.71, 6.18, 2.99, 1.74, 5.68, 3.8, 2.93, 6.07, 4.67, 4.81, -0.24, 2.89, 5.09, -0.26, 3.97, 2.59, 5.08, 5.35, 2.0, 5.21, 6.66, 5.43, 4.07, 3.24, -0.7, 6.37, 7.83, 2.68, 4.75, 3.65, 4.45, 3.55, 3.47, 3.94, 3.95, 3.87, 3.36, 3.49, 3.89, 5.29, 3.64, 4.58, 5.69, 4.3, 2.76, 3.29, 5.31, 4.2, 3.23, 3.0, 3.81, 3.47, 2.92, 4.65, 5.27, 3.02, 1.6, -1.95, 7.7, 2.85, 1.16, 3.22, 4.1, 2.19, 5.03, 2.88, 5.7, 1.25, 3.26, 3.11, 4.04, 3.87, 2.96, 5.06, 4.1, 3.92, 4.39, 7.32, 5.06, 3.31, 3.69, 3.11, 6.29, 5.98, 3.86, 2.6, 8.25, 4.14, 4.4, 9.0, 3.13, 4.94, 8.28, 4.58, 3.79, 3.84, 3.29, 4.0, 4.06, 3.17, 2.88, 0.79, 4.41, 6.05, 3.68, 6.94, 4.7, 4.79, 4.57, 2.3, 5.14, 3.03, 1.33, 2.63, 4.97, 4.19, 7.97, 3.55, 4.15, 2.74, 4.22, 2.42, 7.31, 2.82, 4.61, 4.6, 3.8, 4.11, 2.45, 4.34, 3.99, 1.85, 3.53, 2.95, 2.9, 5.38, 2.71, 2.9, 2.46, 5.49, 2.39, 8.83, 5.51, 1.88, 4.86, 3.07, 4.17, 3.3, 4.12, 4.26, 4.01, 3.95, 2.16, 4.44, 1.3, 4.46, 3.02, 4.66, 2.66, 3.55, 4.3, 5.36, 5.37, 5.38, 3.44, 8.13, 4.31, 5.02, 4.96, 5.08, 5.39, 4.77, 3.46, 1.34, 2.8, 4.21, 4.05, 1.37, 3.55, 5.47, 5.36, 5.4, 3.1, 3.04, 4.4, 7.16, 3.39, 3.31, 2.23, 3.1, 4.63, 3.88, 4.59, 4.95, 5.73, 1.97, 3.32, 3.07, 5.17, 2.91, 1.02, 3.81, 3.69, 3.5, 5.04, 3.54, 5.15, 4.24, 0.61, 3.59, 4.84, 4.24, 5.56, 3.39, 2.37, 4.36, 4.36, 2.77, 4.79, 3.99, 3.72, 7.53, 6.97, 7.64, 6.29, 0.69, 5.72, -1.24, 4.5, 2.44, 2.31, 4.52, 2.54, 4.09, 4.95, 2.16, 7.87, 4.11, 5.11, 3.64, 3.78, 3.98, 3.9, 3.27, 2.48, 4.38, 4.6, 2.48, 2.95, 5.05, 5.1, 2.08, 6.8, 5.49, 4.62, -0.07, 7.54, 5.13, 1.5, 6.32, 5.48, 5.92, 5.0, 4.17, 2.7, 4.96, 6.83, 2.18, 3.08, 3.5, 4.81, 3.04, 2.93, 4.65, 4.24, 3.89, 3.05, 8.0, 1.72, 5.61, 2.28, 2.4, 4.39, 3.66, 4.66, 2.44, 7.83, 4.9, 4.93, 2.54, 3.24, 4.83, 5.48, 4.72, -0.6, 0.95, 0.46, 6.09, 4.67, 7.2, 3.75, 2.06, 4.45, 5.66, 3.21, 2.99, 5.06, 2.81, 6.0, 4.3, 4.88, 2.72, 2.72, 0.59, 4.72, 3.0, 3.12, 4.46, 1.99, 4.81, 3.0, 0.45, 4.36, 3.4], [2.27, 4.21, -3.94, -1.58, 2.15, 1.01, 2.97, 0.67, -0.9, 0.28, 0.69, 0.98, 0.39, 0.94, -2.02, 1.93, 1.0, 3.96, 0.28, 2.02, 5.0, -1.43, 0.75, -0.59, 2.21, 6.11, 0.99, 2.76, 3.05, -0.38, -2.27, -0.24, 3.38, 1.09, 1.9, 4.1, 1.59, -0.52, 1.73, 0.52, 0.1, 0.13, 3.57, -0.31, 0.06, -1.04, 5.03, 1.65, 1.6, 0.87, 1.06, 2.89, 1.2, 1.56, 3.32, -1.52, 0.88, -2.69, 0.1, 0.99, 2.1, -3.19, 0.2, 1.51, -1.73, 2.06, -0.55, 2.14, -1.07, 4.26, 1.3, 1.92, 3.74, 1.85, 0.94, 0.42, 2.91, 3.46, 4.01, 4.8, 2.01, 5.45, 1.18, 3.09, -1.2, 0.39, 1.26, 1.77, 0.45, 2.05, 1.98, 3.03, -1.07, 1.17, -1.11, 1.2, -0.2, 0.75, -1.74, -3.3, -0.59, 2.13, 3.09, -1.4, 1.03, 1.48, -1.73, 2.01, 3.56, 0.25, 1.18, 3.28, 2.22, -1.66, -4.49, 1.61, -0.52, 0.72, 1.4, 0.99, 3.83, 1.13, -0.9, 0.81, 0.26, 0.54, 2.09, 2.86, -0.59, 4.32, 0.17, 0.32, 0.29, 0.46, -1.1, 1.37, 1.39, 2.8, -1.95, 1.75, 1.0, -0.43, 0.35, 2.92, -1.2, 1.51, -0.59, -0.51, 4.57, -1.34, 1.77, 0.07, 3.28, 0.81, 0.84, -0.18, -0.05, 4.95, 1.85, -0.11, 1.23, 1.56, 3.2, 1.04, 0.65, 0.08, -3.91, 1.53, 0.56, 1.14, 2.62, -0.22, -0.41, -1.06, -2.14, 2.05, 3.07, 1.53, 3.33, 0.77, 0.79, 0.15, -0.04, 0.58, 0.2, 2.66, 2.85, 1.24, 3.17, 1.26, 1.15, -0.13, 4.15, -0.55, 2.2, 1.53, -3.41, -1.04, 0.44, -1.22, 3.39, 0.08, 0.54, -0.41, -0.85, -0.17, -2.22, -0.87, 2.17, 1.4, -0.57, 2.73, -2.63, 0.39, 0.57, 1.92, -0.81, -0.88, -1.46, -2.21, -0.09, 1.15, -0.88, 0.6, 0.99, 0.14, -1.81, 1.17, 4.06, -1.22, 1.33, 2.13, 2.3, 1.68, 1.31, 1.79, -0.16, 0.04, 0.89, 3.55, 1.16, 1.88, 1.99, 0.35, -1.09, 3.38, 2.12, 1.57, 0.87, -0.43, -2.32, 1.73, 0.96, 0.77, 1.81, 1.35, 1.48, -0.21, 0.04, 2.69, 1.75, 0.13, 1.45, -1.61, 2.2, 1.32, -0.12, -1.96, 2.29, -0.17, 2.32, 0.85, -1.4, 0.85, -0.9, 0.26, 0.9, 1.65, 1.36, 2.06, 0.88, 4.69, 0.15, 3.91, 0.92, 1.48, 3.61, -2.03, 1.93, 2.5, -0.93, 3.88, 0.04, 3.49, 2.28, 3.7, -0.67, 1.59, -0.58, 0.7, 2.06, 0.65, 2.68, -0.25, -1.21, 2.3, -1.05, 0.82, 1.4, 1.75, 3.98, 2.6, 1.19, 3.63, 2.64, 0.3, -3.56, -2.73, 3.23, 3.27, 1.79, 2.78, 1.7, 0.71, -1.95, -0.51, -1.1, 2.11, -0.66, -0.06, 2.45, 0.02, 2.87, -0.23, 3.51, 0.23, -1.11, 0.34, 1.34, 1.64, 2.56, 0.71, 0.7, -1.19, 0.21, -1.62, 3.02, -0.68, 1.28, 2.0, 0.38, 3.13, 0.28, 1.2, 4.82, 1.1, 2.0, 0.58, 2.02, 3.33, 2.1, 2.43, -0.71, 1.42, 0.08, -0.57, -3.58, 0.73, 2.43, -0.66, 0.99, -0.52, 1.51, 1.75, -0.05, 0.69, 0.34, 0.8, 0.08, 1.81, -0.35, 0.68, -0.07, -2.31, -0.95, 0.02, 3.13, -2.03, -2.67, 0.24, -4.06, -1.73, 0.07, 0.52, -1.27, 1.47, -0.1, 3.93, 2.05, -0.7, 3.06, -1.94, 2.63, 1.07, 0.44, 1.82, 1.86, 0.57, 1.48, -0.26, -0.86, -0.49, 0.14, -0.29, 1.95, 0.43, 2.95, 4.08, 2.44, 1.44, 1.75, 1.2, -1.55, 0.36, -2.08, 1.27, 2.58, 1.4, -0.02, 0.72, 0.27, -0.3, 3.36, 0.62, 0.48, 1.53, -1.32, 0.32, 0.03, 2.28, 1.91, 0.02, 3.03, 0.79, -0.3, 0.41, -3.14, 2.93, 1.75, 3.1, 2.54, 4.45, 1.61, 0.72, -3.21, 1.59, 3.09, 0.58, 1.7, 3.09, 2.37, 0.41, 0.38, 2.9, 1.55, -0.97, 1.14, 1.67, -1.83, 0.4, 4.55, 1.58, 0.29, -0.56, 0.29, 1.88, -1.13, 0.87, -0.8, -0.31, -0.68, 0.29, -0.32, 3.43, -1.81, -0.14, 1.15, -3.24, 1.62, 2.12, 3.9, -0.36, -0.76, -2.98, 1.68, 0.35, 0.15, 3.43, 0.65, 3.12, 0.98, 0.39, 3.45, -0.0, 0.29, 1.24, 2.66, -0.81, 0.81, 2.42, 2.6, -0.8, 2.59, -0.52, 0.02, 1.71, 3.03, 0.74, -3.09, 0.69, 0.62, 3.16, -3.29, 0.71, 0.03, -0.27, 1.87, -2.06, 3.19, 4.36, 3.6, 1.1, 2.94, 3.43, -0.31, 2.61, 1.65, 2.93, 3.02, 2.38, 0.03, 3.47, 1.07, -0.69, 1.63, 1.4, 0.41, 0.02, 1.95, 1.26, -0.09, 0.28, 0.73, 1.7, 2.29, 3.33, 1.64, -0.53, 1.36, 0.86, 2.94, 1.72, 2.18, 4.01, -0.95, 1.47, 1.31, 0.74, 3.09, 0.56, 3.15, 1.61, -0.14, 0.36, -1.3, 1.3, 4.77, 1.48, 1.79, -1.01, 0.05, 2.33, 2.62, 1.84, -2.11, 1.02, 1.51, 3.03, 2.45, 4.26, 0.16, -1.13, -0.6, 0.54, 1.08, 1.4, 0.72, 2.91, -0.6, 2.53]]]
TINC_COUNTS = {"0": [35, 52, 12, 52], "35": [42, 52, 29, 52], "50": [42, 52, 26, 52], "71": [41, 52, 27, 52], "100": [37, 52, 29, 52]}     # hits, n, false alarms, n
TINC_ML = [1.159, 0.706, 0.848, 0.735, 0.404]
TINC_MAP = [1.13, 0.671, 0.833, 0.71, 0.445]       # the posterior modes, as in the analysis
MICHEYL = {0: (1.5, (1.3, 1.85), (0.85, 2.15)), 35: (1.9, (1.75, 2.1), (1.45, 2.4)),
           50: (2.2, (2.05, 2.4), (1.75, 2.65)), 71: (2.4, (2.25, 2.6), (1.95, 2.9)),
           100: (2.5, (2.3, 2.8), (1.85, 3.25))}

def dprime_draws(h, ns, f, nn, seed):
    rng = np.random.default_rng(seed)
    return stats.norm.ppf(rng.beta(h + 1, ns - h + 1, 20000)) - stats.norm.ppf(rng.beta(f + 1, nn - f + 1, 20000))

def violins(ax, x, point, draws):
    for xi, d in zip(x, draws):
        v = ax.violinplot(d, [xi], widths=0.7, showextrema=False, points=200)
        for b in v["bodies"]:
            b.set_facecolor(SOFT); b.set_edgecolor("none"); b.set_alpha(0.55)
    q = np.array([np.percentile(d, [2.5, 12.5, 87.5, 97.5]) for d in draws])
    ax.vlines(x, q[:, 0], q[:, 3], color=ACC, lw=1.0); ax.vlines(x, q[:, 1], q[:, 2], color=ACC, lw=4.5)
    ax.plot(x, point, "o", ms=6.5, mfc="white", mec=ACC, mew=1.6, zorder=4, label="P00")

def trend(ax, x, t, y):
    sl, ic, *_ = stats.linregress(t, y)
    ax.plot(x, ic + sl * np.array(t), color=INK, lw=0.9, ls=(0, (4, 3)), zorder=1)
    return sl

fig, (a, b) = plt.subplots(1, 2, figsize=(10, 3.9))
x = np.arange(5)
violins(a, x, CMR, CMR_DRAWS); trend(a, x, STEPS, CMR)
a.axhline(0, color="0.55", lw=0.7)
a.set_xticks(x, [str(s) for s in STEPS]); a.set_xlim(-0.6, 4.6); a.set_ylim(-6, 15)
a.set_xlabel("Onset step between bands, ΔT (ms)"); a.set_ylabel("Masking release, CMR (dB)")
a.set_title("tcmr: two sessions pooled", loc="left", fontsize=10.5, color=INK)
drop = CMR_DRAWS[0] - CMR_DRAWS[-1]; lo, hi = np.percentile(drop, [2.5, 97.5])
a.text(0.98, 0.97, f"drop from 0 to 40 ms: {CMR[0] - CMR[-1]:.1f} dB\n95%: {lo:.1f} to {hi:.1f}",
       transform=a.transAxes, ha="right", va="top", fontsize=9, color=INK)

delays = [0, 35, 50, 71, 100]
dd = [dprime_draws(*TINC_COUNTS[str(d)], seed=i) for i, d in enumerate(delays)]
for xi, d in zip(x + 0.28, delays):
    _, q75, q95 = MICHEYL[d]
    b.vlines(xi, *q95, color=GREY, lw=0.9); b.vlines(xi, *q75, color=GREY, lw=3.5)
b.plot(x + 0.28, [MICHEYL[d][0] for d in delays], "s", ms=5, mfc="white", mec=GREY, mew=1.4, zorder=4,
       label="Micheyl et al. (2013)")
violins(b, x, TINC_MAP, dd)
sl = trend(b, x, delays, TINC_ML)
b.axhline(0, color="0.55", lw=0.7)
b.set_xticks(x, [str(d) for d in delays]); b.set_xlim(-0.6, 4.8); b.set_ylim(-0.5, 3.5)
b.set_xlabel("A–B onset asynchrony (ms)"); b.set_ylabel("Sensitivity, d′")
b.set_title("tinc: A and B roved together", loc="left", fontsize=10.5, color=INK)
b.text(0.02, 0.97, f"slope {sl * 100:+.2f} d′ per 100 ms", transform=b.transAxes, ha="left", va="top", fontsize=9, color=INK)
b.legend(frameon=False, fontsize=8.5, loc="upper left", bbox_to_anchor=(0, 0.9), handlelength=1.0)
fig.tight_layout(); plt.show()
